# Лабораторная работа №9: Fine-tuning VLM с помощью LLaMA-Factory

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Преобразовать мультимодальные данные в instruction format.
- Сформировать QLoRA YAML и запустить SFT.
- Сравнить base и adapter, включая OOD slice.


## Теоретическая часть

SFT минимизирует autoregressive loss только на целевом ответе. QLoRA хранит base weights в 4-bit NF4 и обучает low-rank update $\Delta W=BA$, что уменьшает память. LLaMA-Factory фиксирует pipeline в YAML-конфигурации.


### Математическая постановка

$$W'=W_q+\frac{\alpha}{r}BA,\quad B\in\mathbb R^{d\times r},\ A\in\mathbb R^{r\times k}.$$ Обучаются только $A,B$; сравнение проводится с той же base model и evaluation set.


### Материалы

- [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
- [QLoRA](https://arxiv.org/abs/2305.14314)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Вход — лицензированный предметный корпус `data/lab9/manifest.jsonl` с независимыми train/validation/ID/OOD. Получение и экспертная проверка данных обязательны; файл не заменяется выдуманным изображением. В Colab установите совместимый LLaMA-Factory из зафиксированного commit и запишите commit в отчёт. CLI запускается непосредственно из ноутбука. Полный режим означает фактическое обучение и сравнение; `FAST_DEV_RUN` обучает два шага только для проверки стека. Оценку общей способности выполните на отдельном общем VQA-наборе с заранее выбранной метрикой.

In [1]:
import subprocess, sys, os
import importlib.metadata as md

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

def _ver(n):
    try: return md.version(n)
    except md.PackageNotFoundError: return None

CU_INDEX = 'https://download.pytorch.org/whl/cu126'
WANT = {'torch': '2.6.0+cu126', 'torchvision': '0.21.0+cu126', 'torchaudio': '2.6.0+cu126', 'bitsandbytes': '0.45.5'}

if all(_ver(k) == v for k, v in WANT.items()):
    print('Окружение уже согласовано:', {k: _ver(k) for k in WANT})
else:
    # 1. torch-набор ставим ОДНОЙ командой, версии согласованы между собой
    pip('--no-cache-dir', 'torch==2.6.0+cu126', 'torchvision==0.21.0+cu126',
        'torchaudio==2.6.0+cu126', '--index-url', CU_INDEX)

    # 2. констрейнт: ни один последующий пакет не сможет подменить torch/torchvision
    with open('constraints.txt', 'w') as f:
        f.write('torch==2.6.0+cu126\ntorchvision==0.21.0+cu126\ntorchaudio==2.6.0+cu126\n')

    # 3. остальное, включая bitsandbytes, - с констрейнтом и БЕЗ --force-reinstall
    #    (иначе bitsandbytes тянет за собой torch с PyPI под CUDA 13 -> ломается torchvision::nms)
    pip('-c', 'constraints.txt', '--upgrade',
        'numpy', 'pandas==2.2.3',
        'transformers>=4.51.0,<=4.57.1', 'tokenizers>=0.21,<0.22',
        'trl>=0.18.0,<=0.24.0', 'peft>=0.14.0,<=0.17.1',
        'accelerate>=1.3.0,<=1.11.0', 'datasets>=2.16.0,<=4.0.0',
        'huggingface-hub<1.0', 'pyyaml', 'matplotlib', 'pillow',
        'bitsandbytes==0.45.5',
        # лёгкие зависимости LLaMA-Factory (сам он ставится с --no-deps)
        'fire', 'omegaconf', 'tyro<0.9.0', 'einops', 'scipy',
        'sentencepiece', 'tiktoken', 'protobuf', 'av', 'packaging')

    print('Пакеты установлены. Сессия перезапускается автоматически; '
          'после перезапуска запустите ноутбук заново (эта ячейка тогда пропустится).')
    os.kill(os.getpid(), 9)   # перезапуск ядра, чтобы подхватить новый torch


Окружение уже согласовано: {'torch': '2.6.0+cu126', 'torchvision': '0.21.0+cu126', 'torchaudio': '2.6.0+cu126', 'bitsandbytes': '0.45.5'}


In [2]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Манифест и независимые разбиения

*Вес: 1 балл.*

**Постановка.** Загрузите ScienceQA функцией prepare_scienceqa. Сформируйте 512 train и по 64 validation, ID, OOD и general примера с изображениями. ID: natural science, классы 1–6; OOD: тот же предмет, классы 7–12; general: другие предметы. Используйте официальные split и исключите повторные изображения по хешу.

**Результат.** manifest.jsonl и ATTRIBUTION.md.

**Критерий принятия.** Количество записей соответствует протоколу; ни одно изображение не используется в двух разбиениях; версия и лицензия источника указаны.


In [3]:
import shutil
from collections import Counter, defaultdict
from PIL import Image
LIMITS = {'train': 8 if FAST_DEV_RUN else 512, 'validation': 4 if FAST_DEV_RUN else 64, 'id_test': 4 if FAST_DEV_RUN else 64,
          'ood_test': 4 if FAST_DEV_RUN else 64, 'general_test': 4 if FAST_DEV_RUN else 64}
TEST_SPLITS = ('id_test', 'ood_test', 'general_test')

manifest_path = Path(prepare_scienceqa('data/lab9', smoke=FAST_DEV_RUN))
records = [json.loads(s) for s in manifest_path.read_text().splitlines() if s.strip()]
grade = lambda r: int(r['grade'].removeprefix('grade'))

# 1. объём по протоколу
assert dict(Counter(r['split'] for r in records)) == LIMITS, Counter(r['split'] for r in records)
# 2. ни одно изображение не используется в двух разбиениях (хеш содержимого и ID)
splits_by_hash = defaultdict(set)
for r in records: splits_by_hash[r['image_sha256']].add(r['split'])
assert all(len(v) == 1 for v in splits_by_hash.values()) and len(splits_by_hash) == len(records)
assert len({r['id'] for r in records}) == len(records)
# 3. определения сдвигов: ID - natural science 1-6, OOD - natural science 7-12, general - другие предметы
for r in records:
    if r['split'] in ('train', 'validation', 'id_test'): assert r['subject'] == 'natural science' and grade(r) <= 6, r['id']
    elif r['split'] == 'ood_test': assert r['subject'] == 'natural science' and grade(r) >= 7, r['id']
    else: assert r['subject'] != 'natural science', r['id']
# 4. целостность изображений: хеш пересчитывается из файла
import hashlib
for r in records:
    im = Image.open(r['image']).convert('RGB')
    assert hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest() == r['image_sha256'], r['id']
# 5. версия и лицензия источника
attr = (manifest_path.parent / 'ATTRIBUTION.md').read_text()
assert SCIENCEQA_REVISION in attr and 'CC-BY-SA-4.0' in attr
shutil.copy(manifest_path, ARTIFACTS / manifest_path.name); shutil.copy(manifest_path.parent / 'ATTRIBUTION.md', ARTIFACTS / 'ATTRIBUTION.md')
save_json('manifest_check.json', {'manifest': str(manifest_path), 'counts': LIMITS, 'unique_images': len(splits_by_hash),
                                  'dataset': SCIENCEQA_ID, 'revision': SCIENCEQA_REVISION, 'license_card': 'CC-BY-SA-4.0',
                                  'fast_dev_run': FAST_DEV_RUN})
print('manifest:', manifest_path.name, dict(Counter(r['split'] for r in records)), '| уникальных изображений:', len(splits_by_hash))
print(attr)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

manifest: manifest.jsonl {'train': 512, 'validation': 64, 'ood_test': 64, 'general_test': 64, 'id_test': 64} | уникальных изображений: 768
ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: derek-thomas/ScienceQA
Revision: f18b0a70359ebfb41f658fd564208d0355b013f4
Dataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/
Original project: https://scienceqa.github.io/
Image subsets and manifests retain attribution; downloaded images are not committed to this course.
ID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.



### Визуальный аудит корпуса

*Вес: 1 балл.*

**Постановка.** Декодируйте изображения и постройте сетку примеров с вопросами и правильными буквами. Подсчитайте распределение ответов по каждому split: возможный перевес одной буквы следует учитывать при интерпретации accuracy.

**Результат.** data_profile.json и data_preview.png.

**Критерий принятия.** Показаны реальные изображения; правильная буква соответствует индексированному варианту; представлены частоты всех букв отдельно по split.


In [4]:
import textwrap, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
by_split = defaultdict(list)
for r in records: by_split[r['split']].append(r)
SPLITS = list(LIMITS)

problems = []
for r in records:
    if not Path(r['image']).is_file(): problems.append((r['id'], 'нет файла'))
    if not r['question'].strip() or not r['answer'].strip(): problems.append((r['id'], 'пустой вопрос/ответ'))
    if not 0 <= r['answer_index'] < len(r['choices']) or r['answer'] != chr(65 + r['answer_index']): problems.append((r['id'], 'буква не соответствует индексу'))
    if f"{r['answer']}. {r['choices'][r['answer_index']]}" not in r['question']: problems.append((r['id'], 'вариант ответа не найден в вопросе'))
assert not problems, problems[:5]

LETTERS = sorted({chr(65 + i) for r in records for i in range(len(r['choices']))})
freq = {s: {l: sum(r['answer'] == l for r in by_split[s]) for l in LETTERS} for s in SPLITS}
profile = {'n_per_split': {s: len(by_split[s]) for s in SPLITS}, 'answer_letter_frequency': freq,
           'majority_letter_share': {s: max(freq[s].values()) / len(by_split[s]) for s in SPLITS},
           'image_side_min_max': (lambda sz: [min(min(x) for x in sz), max(max(x) for x in sz)])([Image.open(r['image']).size for r in records]),
           'note': 'если одна буква встречается чаще, accuracy угадывания выше 1/число вариантов; сравнивайте с majority-базой'}
save_json('data_profile.json', profile)
for s in SPLITS: print(f"{s:13s} n={len(by_split[s]):3d} {freq[s]}  majority-share={profile['majority_letter_share'][s]:.2f}")

fig, axes = plt.subplots(len(SPLITS), 4, figsize=(13, 3.0 * len(SPLITS)))
for i, s in enumerate(SPLITS):
    for j in range(4):
        ax = axes[i, j]; ax.axis('off')
        if j < len(by_split[s]):
            r = by_split[s][j]; ax.imshow(Image.open(r['image']).convert('RGB'))
            ax.set_title(f"{s} | ответ {r['answer']}\n" + textwrap.fill(r['question'].split('\n')[0], 38)[:120], fontsize=7)
plt.tight_layout(); plt.savefig(ARTIFACTS / 'data_preview.png', dpi=110); plt.show()

train         n=512 {'A': 209, 'B': 230, 'C': 72, 'D': 1, 'E': 0}  majority-share=0.45
validation    n= 64 {'A': 24, 'B': 32, 'C': 8, 'D': 0, 'E': 0}  majority-share=0.50
id_test       n= 64 {'A': 26, 'B': 27, 'C': 11, 'D': 0, 'E': 0}  majority-share=0.42
ood_test      n= 64 {'A': 25, 'B': 27, 'C': 9, 'D': 3, 'E': 0}  majority-share=0.42
general_test  n= 64 {'A': 13, 'B': 21, 'C': 17, 'D': 13, 'E': 0}  majority-share=0.33


### Экспорт мультимодального датасета

*Вес: 1 балл.*

**Постановка.** Преобразуйте записи в ShareGPT: сообщение пользователя содержит один маркер <image> и вопрос с вариантами, ответ ассистента — правильную букву. Зарегистрируйте train и validation в dataset_info.json.

**Результат.** train.json, validation.json и dataset_info.json.

**Критерий принятия.** Пути изображений разрешаются; роли и столбцы соответствуют формату LLaMA-Factory; test не попал в обучение.


In [5]:
LF_DIR = Path('data/lab9/llamafactory').resolve(); LF_DIR.mkdir(parents=True, exist_ok=True)

def to_sharegpt(r):
    assert '<image>' not in r['question']
    return {'messages': [{'role': 'user', 'content': '<image>' + r['question']},          # ровно один маркер <image>
                         {'role': 'assistant', 'content': r['answer']}],                  # ответ - правильная буква
            'images': [str(Path(r['image']).resolve())]}

for split, fname in (('train', 'train.json'), ('validation', 'validation.json')):
    (LF_DIR / fname).write_text(json.dumps([to_sharegpt(r) for r in by_split[split]], ensure_ascii=False, indent=1))
sharegpt_cols = {'messages': 'messages', 'images': 'images'}
sharegpt_tags = {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant'}
dataset_info = {f'lab9_{s}': {'file_name': f'{s}.json', 'formatting': 'sharegpt', 'columns': sharegpt_cols, 'tags': sharegpt_tags}
                for s in ('train', 'validation')}
(LF_DIR / 'dataset_info.json').write_text(json.dumps(dataset_info, indent=2))

# проверки формата и отсутствия утечки test
test_hashes = {r['image_sha256'] for s in TEST_SPLITS for r in by_split[s]}
test_ids = {r['id'] for s in TEST_SPLITS for r in by_split[s]}
test_paths = {str(Path(r['image']).resolve()) for s in TEST_SPLITS for r in by_split[s]}
for fname, split in (('train.json', 'train'), ('validation.json', 'validation')):
    data = json.loads((LF_DIR / fname).read_text()); assert len(data) == LIMITS[split]
    for ex, r in zip(data, by_split[split]):
        assert [m['role'] for m in ex['messages']] == ['user', 'assistant']
        assert ex['messages'][0]['content'].count('<image>') == 1 and len(ex['images']) == 1 and Path(ex['images'][0]).is_file()
        assert ex['images'][0] not in test_paths and r['image_sha256'] not in test_hashes and r['id'] not in test_ids
for name, spec in dataset_info.items():
    assert (LF_DIR / spec['file_name']).is_file() and spec['formatting'] == 'sharegpt'
for f in ('train.json', 'validation.json', 'dataset_info.json'): shutil.copy(LF_DIR / f, ARTIFACTS / f)
save_json('export_check.json', {'dir': str(LF_DIR), 'train': LIMITS['train'], 'validation': LIMITS['validation'],
                                'test_in_training_data': False, 'registered': list(dataset_info)})
print(json.dumps(dataset_info, indent=1)); print(json.dumps(json.loads((LF_DIR / 'train.json').read_text())[0], ensure_ascii=False, indent=1)[:700])

{
 "lab9_train": {
  "file_name": "train.json",
  "formatting": "sharegpt",
  "columns": {
   "messages": "messages",
   "images": "images"
  },
  "tags": {
   "role_tag": "role",
   "content_tag": "content",
   "user_tag": "user",
   "assistant_tag": "assistant"
  }
 },
 "lab9_validation": {
  "file_name": "validation.json",
  "formatting": "sharegpt",
  "columns": {
   "messages": "messages",
   "images": "images"
  },
  "tags": {
   "role_tag": "role",
   "content_tag": "content",
   "user_tag": "user",
   "assistant_tag": "assistant"
  }
 }
}
{
 "messages": [
  {
   "role": "user",
   "content": "<image>Which of these organisms contains matter that was once part of the phytoplankton?\nA. black rockfish\nB. sea otter\nReturn only the letter of the correct option."
  },
  {
   "role": "assistant",
   "content": "A"
  }
 ],
 "images": [
  "/content/data/lab9/7059b2a9d6213ab2b6bc531c6adaa60811f86552fb2ac3b6acf1244b0485c5b9.png"
 ]
}


### Конфигурация QLoRA SFT

*Вес: 1 балл.*

**Постановка.** Настройте Qwen2.5-VL-3B-Instruct, NF4, LoRA rank 8 и alpha 16, batch 1 с accumulation 16, learning rate 1e-4 и 100 шагов. Включите validation каждые 20 шагов.

**Результат.** sft.yaml.

**Критерий принятия.** YAML содержит реальные идентификаторы данных и модели, включает do_train/do_eval и сохраняет checkpoint; полный режим не ограничен двумя smoke-шагами.


In [6]:
import yaml
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
IMAGE_MAX_PIXELS = 262144                                  # общий предел разрешения для обучения и инференса
MAX_STEPS = 2 if FAST_DEV_RUN else 100
EVAL_STEPS = SAVE_STEPS = 1 if FAST_DEV_RUN else 20        # validation и checkpoint каждые 20 шагов (smoke: каждый шаг)
OUT_DIR = (ARTIFACTS / ('lab9_adapter_smoke' if FAST_DEV_RUN else 'lab9_adapter')).resolve()
YAML_PATH = ARTIFACTS / 'sft.yaml'

def build_cfg(resume=None):
    cfg = {
        # model
        'model_name_or_path': MODEL_ID, 'image_max_pixels': IMAGE_MAX_PIXELS, 'trust_remote_code': True,
        # method: QLoRA (NF4 4-bit + LoRA r=8, alpha=16)
        'stage': 'sft', 'do_train': True, 'do_eval': True, 'finetuning_type': 'lora', 'lora_rank': 8, 'lora_alpha': 16,
        'lora_dropout': 0.05, 'lora_target': 'all', 'quantization_bit': 4, 'quantization_method': 'bnb', 'quantization_type': 'nf4',
        'double_quantization': True,
        # dataset
        'dataset': 'lab9_train', 'eval_dataset': 'lab9_validation', 'dataset_dir': str(LF_DIR), 'template': 'qwen2_vl',
        'cutoff_len': 2048, 'overwrite_cache': True, 'preprocessing_num_workers': 4,
        # output
        'output_dir': str(OUT_DIR), 'logging_steps': 5, 'save_steps': SAVE_STEPS, 'save_only_model': False,   # False: сохраняем и состояние оптимизатора
        'plot_loss': True, 'report_to': 'none',
        # train
        'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 16, 'learning_rate': 1.0e-4, 'max_steps': MAX_STEPS,
        'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.1, 'fp16': True, 'bf16': False, 'seed': SEED, 'ddp_timeout': 180000000,
        # eval
        'per_device_eval_batch_size': 1, 'eval_strategy': 'steps', 'eval_steps': EVAL_STEPS,
    }
    if resume is not None: cfg['resume_from_checkpoint'] = str(resume)
    return cfg

def write_sft_yaml(resume=None):
    cfg = build_cfg(resume); YAML_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True)); return cfg

cfg = write_sft_yaml()
assert cfg['model_name_or_path'] == 'Qwen/Qwen2.5-VL-3B-Instruct' and cfg['quantization_bit'] == 4
assert (cfg['lora_rank'], cfg['lora_alpha'], cfg['per_device_train_batch_size'], cfg['gradient_accumulation_steps']) == (8, 16, 1, 16)
assert cfg['learning_rate'] == 1e-4 and cfg['do_train'] and cfg['do_eval'] and cfg['save_steps'] > 0
assert cfg['max_steps'] == (2 if FAST_DEV_RUN else 100) and (FAST_DEV_RUN or cfg['eval_steps'] == 20)
for key in ('dataset', 'eval_dataset'): assert cfg[key] in dataset_info and (LF_DIR / dataset_info[cfg[key]]['file_name']).is_file()
print(YAML_PATH.read_text())

model_name_or_path: Qwen/Qwen2.5-VL-3B-Instruct
image_max_pixels: 262144
trust_remote_code: true
stage: sft
do_train: true
do_eval: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target: all
quantization_bit: 4
quantization_method: bnb
quantization_type: nf4
double_quantization: true
dataset: lab9_train
eval_dataset: lab9_validation
dataset_dir: /content/data/lab9/llamafactory
template: qwen2_vl
cutoff_len: 2048
overwrite_cache: true
preprocessing_num_workers: 4
output_dir: /content/artifacts/lab9_adapter
logging_steps: 5
save_steps: 20
save_only_model: false
plot_loss: true
report_to: none
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 0.0001
max_steps: 100
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
bf16: false
seed: 42
ddp_timeout: 180000000
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 20



### Возобновление обучения

*Вес: 1 балл.*

**Постановка.** Реализуйте выбор последнего checkpoint по числовому global step. Отсутствие checkpoint означает новый запуск; существующий должен принадлежать той же конфигурации. Объясните отличие загрузки весов от восстановления состояния оптимизатора.

**Результат.** resume_protocol.json и обновлённый sft.yaml.

**Критерий принятия.** Выбор не основан на лексикографическом порядке; зафиксированы выбранный путь либо null и режим нового/возобновляемого запуска.


In [7]:
import re, tempfile
CKPT_RE = re.compile(r'^checkpoint-(\d+)$')

def find_latest_checkpoint(out_dir):
    """Последний checkpoint по ЧИСЛОВОМУ global step (checkpoint-100 > checkpoint-20 > checkpoint-9); нет checkpoint -> None."""
    found = []
    for d in Path(out_dir).glob('checkpoint-*'):
        m = CKPT_RE.match(d.name)
        if m and d.is_dir() and (d / 'trainer_state.json').is_file():
            step = json.loads((d / 'trainer_state.json').read_text()).get('global_step')
            assert step == int(m.group(1)), f'{d}: global_step в trainer_state ({step}) не совпадает с именем'
            found.append((int(m.group(1)), d))
    return max(found, key=lambda t: t[0])[1] if found else None

def config_fingerprint(c):
    c = {k: v for k, v in c.items() if k != 'resume_from_checkpoint'}      # resume-путь не часть конфигурации обучения
    return hashlib.sha256(json.dumps(c, sort_keys=True, default=str).encode()).hexdigest()

FP_FILE = OUT_DIR / 'config_fingerprint.json'
def plan_run():
    base_cfg = build_cfg(); fp = config_fingerprint(base_cfg)
    ck = find_latest_checkpoint(OUT_DIR)
    if ck is None:
        OUT_DIR.mkdir(parents=True, exist_ok=True); FP_FILE.write_text(json.dumps({'fingerprint': fp}))
        return 'new', None, fp
    recorded = json.loads(FP_FILE.read_text())['fingerprint'] if FP_FILE.exists() else None
    if recorded != fp:
        raise RuntimeError(f'{ck} создан с другой конфигурацией (отпечаток {recorded} != {fp}); возобновление запрещено')
    return 'resume', ck, fp

# самопроверка правила выбора: лексикографически '9' > '20' > '100', а числово - наоборот
with tempfile.TemporaryDirectory() as t:
    for n in (9, 20, 100):
        (Path(t) / f'checkpoint-{n}').mkdir(); (Path(t) / f'checkpoint-{n}' / 'trainer_state.json').write_text(json.dumps({'global_step': n}))
    assert find_latest_checkpoint(t).name == 'checkpoint-100' and sorted(['checkpoint-9', 'checkpoint-20', 'checkpoint-100'])[-1] == 'checkpoint-9'
    assert find_latest_checkpoint(Path(t) / 'empty') is None

run_mode, resume_ckpt, fingerprint = plan_run()
cfg = write_sft_yaml(resume_ckpt)                              # sft.yaml обновлён: resume_from_checkpoint добавляется только при возобновлении
step = json.loads((resume_ckpt / 'trainer_state.json').read_text())['global_step'] if resume_ckpt else None
save_json('resume_protocol.json', {
    'mode': run_mode, 'resume_from_checkpoint': str(resume_ckpt) if resume_ckpt else None, 'global_step': step,
    'selection_rule': 'максимум по числу в имени checkpoint-<global_step>, проверка совпадения с trainer_state.json; не лексикографический порядок',
    'config_fingerprint': fingerprint, 'same_config_required': True,
    'weights_vs_state': ('Загрузка весов (adapter_model.safetensors / from_pretrained) восстанавливает лишь параметры модели: оптимизатор, '
                         'lr-scheduler, RNG и счётчик шагов начинаются с нуля. resume_from_checkpoint восстанавливает также optimizer.pt, scheduler.pt, '
                         'rng_state и trainer_state.json (global_step), поэтому прогон продолжается так, как будто его не прерывали. '
                         'Для этого нужен save_only_model: false.')})
print('режим:', run_mode, '| checkpoint:', resume_ckpt, '| global_step:', step)

режим: new | checkpoint: None | global_step: None


### Обучение в LLaMA-Factory

*Вес: 1 балл.*

**Постановка.** Выполните llamafactory-cli train с сохранением полного лога. Проверьте фактически выполненное число шагов и наличие обученного адаптера.

**Результат.** lab9_adapter, train.log и version.json.

**Критерий принятия.** Процесс завершился без ошибки; сохранены adapter_model.safetensors и trainer_state.json; запуск с FAST_DEV_RUN=True не принимается за полный эксперимент.


In [10]:
import os
import json
import time
import subprocess
import sys
from pathlib import Path
import importlib.metadata as md
import importlib

LF_REF = os.getenv('LLAMAFACTORY_REF', 'v0.9.4')
LF_SRC = Path('LLaMA-Factory').resolve()


def ensure_llamafactory():
    if not LF_SRC.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/hiyouga/LLaMA-Factory.git', str(LF_SRC)],
            check=True,
        )
    subprocess.run(['git', '-C', str(LF_SRC), 'fetch', '--tags', '--force'], check=True)

    ok = False
    for ref in (f'refs/tags/{LF_REF}', LF_REF):
        r = subprocess.run(
            ['git', '-C', str(LF_SRC), 'rev-parse', '--verify', ref],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if r.returncode == 0:
            ok = True
            break
    if not ok:
        tags = subprocess.check_output(
            ['git', '-C', str(LF_SRC), 'tag', '-l'], text=True
        ).strip().splitlines()
        raise RuntimeError(
            f"LLaMA-Factory ref '{LF_REF}' не найден. "
            f"Доступные теги (последние 10): {tags[-10:]}. "
            f"Задайте LLAMAFACTORY_REF или исправьте LF_REF."
        )

    subprocess.run(['git', '-C', str(LF_SRC), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(LF_SRC), 'clean', '-fd'], check=True)
    subprocess.run(['git', '-C', str(LF_SRC), 'checkout', LF_REF], check=True)

    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-factory'],
        check=False,
    )

    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', str(LF_SRC), '--no-deps'],
        check=True,
    )

    return subprocess.check_output(
        ['git', '-C', str(LF_SRC), 'rev-parse', 'HEAD'], text=True
    ).strip()


def ver(n):
    try:
        return md.version(n)
    except Exception:
        return None


def trained_ok(d):
    ts = Path(d) / 'trainer_state.json'
    return (
        ts.is_file()
        and json.loads(ts.read_text())['global_step'] >= MAX_STEPS
        and (Path(d) / 'adapter_model.safetensors').is_file()
    )


commit = ensure_llamafactory()

_src = str(LF_SRC / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
importlib.invalidate_caches()

import bitsandbytes as bnb
print('bitsandbytes OK:', bnb.__version__)

import torch
import llamafactory
import inspect

save_json('version.json', {
    'llamafactory_ref': LF_REF,
    'llamafactory_commit': commit,
    'llamafactory': ver('llamafactory'),
    'llamafactory_path': inspect.getfile(llamafactory),
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': ver('transformers'),
    'numpy': ver('numpy'),
    'peft': ver('peft'),
    'trl': ver('trl'),
    'bitsandbytes': ver('bitsandbytes'),
    'datasets': ver('datasets'),
    'accelerate': ver('accelerate'),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'fast_dev_run': FAST_DEV_RUN,
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
})

LOG = ARTIFACTS / 'train.log'
if trained_ok(OUT_DIR):
    print('Обучение уже завершено, адаптер найден:', OUT_DIR)
else:
    assert torch.cuda.is_available(), 'Нужен GPU (T4 и выше) для QLoRA-обучения'
    with LOG.open('a' if run_mode == 'resume' else 'w') as lf:
        proc = subprocess.Popen(
            ['llamafactory-cli', 'train', str(YAML_PATH)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        )
        for line in proc.stdout:
            lf.write(line)
            if any(k in line for k in ("'loss'", 'eval_loss', 'Error', 'Traceback', 'Saving model')):
                print(line.rstrip()[:200])
        rc = proc.wait()
    assert rc == 0, f'llamafactory-cli завершился с кодом {rc}; см. {LOG}'

assert trained_ok(OUT_DIR), 'нет adapter_model.safetensors / trainer_state.json с нужным числом шагов'
ts = json.loads((OUT_DIR / 'trainer_state.json').read_text())
evals = [(h['step'], h['eval_loss']) for h in ts['log_history'] if 'eval_loss' in h]
save_json('train_summary.json', {
    'global_step': ts['global_step'],
    'max_steps_expected': MAX_STEPS,
    'full_experiment': not FAST_DEV_RUN,
    'train_losses': [(h['step'], h['loss']) for h in ts['log_history'] if 'loss' in h],
    'eval_losses': evals,
    'adapter': str(OUT_DIR / 'adapter_model.safetensors'),
    'log': str(LOG),
})
print('шагов выполнено:', ts['global_step'], '| eval_loss по шагам:', evals)
if FAST_DEV_RUN:
    print('FAST_DEV_RUN: это проверка стека (2 шага), НЕ полный эксперимент.')

bitsandbytes OK: 0.45.5
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: object cannot be re-sized
Traceback (most recent call last):
BufferError: Existing exports of data: ob

### Детерминированный инференс

*Вес: 1 балл.*

**Постановка.** Загрузите исходную модель и процессор. Реализуйте predict: передайте реальное изображение, примените chat template, декодируйте только продолжение после prompt, отключите sampling.

**Результат.** Функция predict.

**Критерий принятия.** Ответ не включает входной вопрос; base и SFT используют один процессор, одинаковые входы и max_new_tokens.


In [11]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
MAX_NEW_TOKENS = 8                                                   # одинаково для base и SFT
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=IMAGE_MAX_PIXELS)      # один процессор для обеих моделей
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto',
                                                                torch_dtype=torch.float16).eval()

def predict(model, image_path, question):
    """Детерминированный ответ: реальное изображение + chat template, sampling выключен, декодируется только продолжение."""
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image = Image.open(image_path).convert('RGB')
    inputs = processor(text=[prompt], images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0].strip()

def extract_letter(text):
    """Метрика, выбранная заранее: ответ должен начинаться с заглавной буквы варианта (A, A., (A), 'A. текст').
    Иной формат ('The answer is A', пусто) - ошибка формата и засчитывается как неверный ответ."""
    m = re.match(r'^\(?([A-E])\)?(?:[.:)]|\s|$)', text.strip())
    return m.group(1) if m else None

# проверка на примере validation (не test): ответ не содержит вопроса и воспроизводим
ex = by_split['validation'][0]
a1, a2 = predict(base_model, ex['image'], ex['question']), predict(base_model, ex['image'], ex['question'])
assert a1 == a2 and ex['question'].split('\n')[0] not in a1
print('пример:', repr(a1), '| буква:', extract_letter(a1), '| эталон:', ex['answer'])

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


пример: 'B' | буква: B | эталон: B


### Исходное качество без адаптера

*Вес: 1 балл.*

**Постановка.** Получите предсказания base на всех заранее отделённых ID/OOD/general изображениях. Для каждого ответа сохраните ID, split и эталон.

**Результат.** base_predictions.jsonl.

**Критерий принятия.** Каждый тестовый ID представлен ровно один раз; предсказания получены моделью, не подставлены из правильных ответов.


In [12]:
test_recs = [r for s in TEST_SPLITS for r in by_split[s]]
BASE_PATH = ARTIFACTS / 'base_predictions.jsonl'

def load_jsonl(path):
    path = Path(path)
    return [json.loads(s) for s in path.read_text().splitlines() if s.strip()] if path.exists() else []

def run_predictions(model, recs, path, log_every=25):
    """Возобновляемый инференс: каждая запись пишется сразу; пары определяются по ID. Эталон сохраняется рядом, но модели не передаётся."""
    done = {r['id'] for r in load_jsonl(path)}
    with Path(path).open('a') as f:
        for n, r in enumerate([r for r in recs if r['id'] not in done], 1):
            resp = predict(model, r['image'], r['question'])
            f.write(json.dumps({'id': r['id'], 'split': r['split'], 'gold': r['answer'], 'response': resp,
                                'pred_letter': extract_letter(resp)}, ensure_ascii=False) + '\n'); f.flush()
            if n % log_every == 0: print(f'  {n}')

run_predictions(base_model, test_recs, BASE_PATH)
base_rows = load_jsonl(BASE_PATH)
assert len(base_rows) == len(test_recs) and {r['id'] for r in base_rows} == {r['id'] for r in test_recs}
assert len({r['id'] for r in base_rows}) == len(base_rows)           # каждый ID ровно один раз
print(Counter(r['split'] for r in base_rows), '| ошибок формата:', sum(r['pred_letter'] is None for r in base_rows))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  25


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

...

  150


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  175


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

Counter({'id_test': 64, 'ood_test': 64, 'general_test': 64}) | ошибок формата: 0


### Парные ответы после SFT

*Вес: 1 балл.*

**Постановка.** Подключите обученный LoRA и повторите инференс на тех же ID без изменения вопросов и декодирования. Сохраните base и SFT рядом.

**Результат.** predictions.jsonl.

**Критерий принятия.** Пары сопоставлены по ID, а не по случайному порядку; на каждом тесте есть ответы обеих моделей.


In [13]:
from peft import PeftModel
assert trained_ok(OUT_DIR)
sft_model = PeftModel.from_pretrained(base_model, str(OUT_DIR)).eval()   # тот же processor, те же вопросы и декодирование
SFT_PATH = ARTIFACTS / 'sft_predictions.jsonl'
run_predictions(sft_model, test_recs, SFT_PATH)

base_by_id = {r['id']: r for r in load_jsonl(BASE_PATH)}; sft_by_id = {r['id']: r for r in load_jsonl(SFT_PATH)}
assert set(base_by_id) == set(sft_by_id) == {r['id'] for r in test_recs}
with (ARTIFACTS / 'predictions.jsonl').open('w') as f:                   # пары сопоставлены по ID
    for r in test_recs:
        b, s = base_by_id[r['id']], sft_by_id[r['id']]
        assert b['gold'] == s['gold'] == r['answer'] and b['split'] == s['split'] == r['split']
        f.write(json.dumps({'id': r['id'], 'split': r['split'], 'gold': r['answer'],
                            'base_response': b['response'], 'base_pred': b['pred_letter'],
                            'sft_response': s['response'], 'sft_pred': s['pred_letter']}, ensure_ascii=False) + '\n')
pairs = load_jsonl(ARTIFACTS / 'predictions.jsonl')
print({s: sum(p['split'] == s for p in pairs) for s in TEST_SPLITS})

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  25


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

...

  150


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

  175


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

{'id_test': 64, 'ood_test': 64, 'general_test': 64}


### Качество и перенос

*Вес: 1 балл.*

**Постановка.** Посчитайте exact match правильной буквы отдельно по трём тестам. Укажите число примеров и изменение после SFT. Объясните, почему улучшение ID не гарантирует улучшения OOD.

**Результат.** evaluation.json и предметный вывод.

**Критерий принятия.** В знаменателе все примеры соответствующего теста; ошибки формата не исключены; вывод различает ID, OOD и general.


In [14]:
def split_metrics(rows):
    n = len(rows); b = np.array([r['base_pred'] == r['gold'] for r in rows]); s = np.array([r['sft_pred'] == r['gold'] for r in rows])  # None -> неверно
    rng_ = np.random.RandomState(SEED); d = []
    for _ in range(2000):
        ix = rng_.randint(0, n, n); d.append(s[ix].mean() - b[ix].mean())
    letters = Counter(r['gold'] for r in rows)
    return {'n': n, 'exact_match_base': float(b.mean()), 'exact_match_sft': float(s.mean()), 'delta': float(s.mean() - b.mean()),
            'delta_bootstrap_95ci': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            'base_wrong_sft_right': int((~b & s).sum()), 'base_right_sft_wrong': int((b & ~s).sum()),
            'format_errors_base': int(sum(r['base_pred'] is None for r in rows)), 'format_errors_sft': int(sum(r['sft_pred'] is None for r in rows)),
            'majority_letter_share': max(letters.values()) / n}

evaluation = {s: split_metrics([p for p in pairs if p['split'] == s]) for s in TEST_SPLITS}
for s, m in evaluation.items():
    assert m['n'] == LIMITS[s]                                           # в знаменателе все примеры теста
    print(f"{s:13s} n={m['n']:3d}  EM base={m['exact_match_base']:.3f} -> SFT={m['exact_match_sft']:.3f}  Δ={m['delta']:+.3f} "
          f"CI95[{m['delta_bootstrap_95ci'][0]:+.3f};{m['delta_bootstrap_95ci'][1]:+.3f}]  fmt-err {m['format_errors_base']}->{m['format_errors_sft']}")

d_id, d_ood, d_gen = (evaluation[s]['delta'] for s in TEST_SPLITS)
verdict = [f"ID (natural science, 1-6 классы): {d_id:+.3f}; OOD (7-12 классы): {d_ood:+.3f}; general (другие предметы): {d_gen:+.3f}."]
if d_id > 0 and d_ood < d_id: verdict.append('Прирост на OOD меньше, чем на ID: адаптер лучше подогнан под распределение обучения.')
elif d_id > 0: verdict.append('Прирост на OOD не меньше, чем на ID: перенос в пределах предмета сохраняется.')
else: verdict.append('Прироста на ID нет: SFT не дал измеримого улучшения на 64 примерах.')
verdict.append('general ' + ('ухудшился: возможна потеря общей способности (forgetting).' if d_gen < 0 else 'не ухудшился (в пределах шума выборки).'))
explanation = ('Улучшение ID не гарантирует улучшения OOD: LoRA-адаптер минимизирует loss только на обучающем распределении (естественные науки 1-6 классов), '
               'и часть прироста может быть выучена как формат ответа (одна буква) и поверхностные паттерны, специфичные для этого распределения. '
               'Задачи 7-12 классов требуют иных знаний/рассуждений, поэтому там эти паттерны могут не работать. '
               'Дополнительно на n=64 доверительные интервалы широки (±10-12 п.п.), а одна буква может встречаться чаще остальных (см. majority_letter_share).')
save_json('evaluation.json', {'metric': 'exact match правильной буквы (формат-ошибки засчитаны неверными, знаменатель - все примеры теста)',
                              'full_experiment': not FAST_DEV_RUN, 'splits': evaluation, 'conclusion': verdict, 'explanation': explanation})
print('\n'.join(verdict)); print(explanation)

id_test       n= 64  EM base=0.875 -> SFT=0.906  Δ=+0.031 CI95[-0.062;+0.125]  fmt-err 0->0
ood_test      n= 64  EM base=0.594 -> SFT=0.719  Δ=+0.125 CI95[+0.031;+0.234]  fmt-err 0->0
general_test  n= 64  EM base=0.859 -> SFT=0.859  Δ=+0.000 CI95[+0.000;+0.000]  fmt-err 0->0
ID (natural science, 1-6 классы): +0.031; OOD (7-12 классы): +0.125; general (другие предметы): +0.000.
Прирост на OOD не меньше, чем на ID: перенос в пределах предмета сохраняется.
general не ухудшился (в пределах шума выборки).
Улучшение ID не гарантирует улучшения OOD: LoRA-адаптер минимизирует loss только на обучающем распределении (естественные науки 1-6 классов), и часть прироста может быть выучена как формат ответа (одна буква) и поверхностные паттерны, специфичные для этого распределения. Задачи 7-12 классов требуют иных знаний/рассуждений, поэтому там эти паттерны могут не работать. Дополнительно на n=64 доверительные интервалы широки (±10-12 п.п.), а одна буква может встречаться чаще остальных (см. majori

### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [15]:
# Гипотеза (сформулирована до запуска; test для выбора гипотезы не использовался):
# H: прирост exact match после SFT на буквенных ответах объясняется главным образом исправлением ФОРМАТА ответа
#    (меньше ответов не в виде одной буквы), а не знаниями: >=50% прироста строгой метрики сохраняется,
#    если оценивать «мягкой» метрикой, прощающей формат (любая отдельная заглавная буква A-E в ответе).
# Протокол: дополнительные реальные входы = split validation (64 изображения, не входят в test; использовались лишь для мониторинга loss).
VAL_RECS = by_split['validation']
lenient = lambda t: (re.search(r'\b([A-E])\b', t) or [None, None])[1]
def predict_both(recs, path):
    done = {r['id'] for r in load_jsonl(path)}
    with Path(path).open('a') as f:
        for r in [r for r in recs if r['id'] not in done]:
            b_resp = None
            with sft_model.disable_adapter(): b_resp = predict(sft_model, r['image'], r['question'])     # base = тот же процессор/декодирование
            s_resp = predict(sft_model, r['image'], r['question'])
            f.write(json.dumps({'id': r['id'], 'gold': r['answer'], 'base_response': b_resp, 'sft_response': s_resp}, ensure_ascii=False) + '\n'); f.flush()
BONUS_PATH = ARTIFACTS / 'bonus_validation_predictions.jsonl'
predict_both(VAL_RECS, BONUS_PATH); vrows = load_jsonl(BONUS_PATH); assert len(vrows) == len(VAL_RECS)

def decompose(rows, bk='base_response', sk='sft_response'):
    em = lambda k, f: float(np.mean([f(r[k]) == r['gold'] for r in rows]))
    strict_b, strict_s, len_b, len_s = em(bk, extract_letter), em(sk, extract_letter), em(bk, lenient), em(sk, lenient)
    d_strict, d_len = strict_s - strict_b, len_s - len_b
    return {'n': len(rows), 'strict_base': strict_b, 'strict_sft': strict_s, 'lenient_base': len_b, 'lenient_sft': len_s,
            'delta_strict': d_strict, 'delta_lenient': d_len,
            'format_err_base': float(np.mean([extract_letter(r[bk]) is None for r in rows])),
            'format_err_sft': float(np.mean([extract_letter(r[sk]) is None for r in rows])),
            'share_of_gain_explained_by_format': (d_strict - d_len) / d_strict if d_strict > 0 else None}
val_d = decompose(vrows)
test_d = decompose([{'gold': p['gold'], 'base_response': p['base_response'], 'sft_response': p['sft_response']} for p in pairs])
f = lambda x: 'н/д' if x is None else f'{x:.3f}'
share = val_d['share_of_gain_explained_by_format']
verdict = ('прироста строгой метрики на validation нет - гипотеза не проверяема' if share is None else
           ('гипотеза подтверждена' if share >= 0.5 else 'гипотеза НЕ подтверждена'))
md = f"""# Бонус: SFT исправляет формат ответа или знания?

## Гипотеза
>=50% прироста строгого exact match после SFT приходится на исправление формата (разница между строгой и «мягкой» метрикой).

## Протокол
- Дополнительные реальные входы: validation ({val_d['n']} изображений ScienceQA), не входящие в test; отложенный test не использовался для выбора гипотезы.
- base = та же квантованная модель с отключённым адаптером (`disable_adapter`), SFT = с адаптером; один processor, max_new_tokens={MAX_NEW_TOKENS}, sampling выключен.
- Строгая метрика: ответ начинается с буквы; мягкая: любая отдельная заглавная A-E в тексте.

## Наблюдения
| набор | strict base→SFT | lenient base→SFT | ошибки формата base→SFT | доля прироста от формата |
|---|---|---|---|---|
| validation (n={val_d['n']}) | {f(val_d['strict_base'])}→{f(val_d['strict_sft'])} | {f(val_d['lenient_base'])}→{f(val_d['lenient_sft'])} | {f(val_d['format_err_base'])}→{f(val_d['format_err_sft'])} | {f(share)} |
| test, все 3 набора (n={test_d['n']}) | {f(test_d['strict_base'])}→{f(test_d['strict_sft'])} | {f(test_d['lenient_base'])}→{f(test_d['lenient_sft'])} | {f(test_d['format_err_base'])}→{f(test_d['format_err_sft'])} | {f(test_d['share_of_gain_explained_by_format'])} |

## Вывод
Результат: {verdict}. Ограничения: мягкая метрика может засчитывать случайные буквы в тексте; n=64 даёт широкие доверительные интервалы; validation использовался для мониторинга обучения, поэтому он «чище» test, но не полностью независим от процесса обучения.
Расхождения с основным экспериментом (test) объясняются разной сложностью и составом наборов (ID/OOD/general) и малым n.
"""
(ARTIFACTS / 'bonus.md').write_text(md); save_json('bonus_stats.json', {'validation': val_d, 'test_pooled': test_d, 'verdict': verdict}); print(md)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

# Бонус: SFT исправляет формат ответа или знания?

## Гипотеза
>=50% прироста строгого exact match после SFT приходится на исправление формата (разница между строгой и «мягкой» метрикой).

## Протокол
- Дополнительные реальные входы: validation (64 изображений ScienceQA), не входящие в test; отложенный test не использовался для выбора гипотезы.
- base = та же квантованная модель с отключённым адаптером (`disable_adapter`), SFT = с адаптером; один processor, max_new_tokens=8, sampling выключен.
- Строгая метрика: ответ начинается с буквы; мягкая: любая отдельная заглавная A-E в тексте.

## Наблюдения
| набор | strict base→SFT | lenient base→SFT | ошибки формата base→SFT | доля прироста от формата |
|---|---|---|---|---|
| validation (n=64) | 0.797→0.828 | 0.797→0.828 | 0.000→0.000 | 0.000 |
| test, все 3 набора (n=192) | 0.776→0.828 | 0.776→0.828 | 0.000→0.000 | 0.000 |

## Вывод
Результат: гипотеза НЕ подтверждена. Ограничения: мягкая метрика может засчитывать случайные буквы в тексте;

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

## Инструкция повторного запуска

1. Colab: GPU T4 (16 ГБ). Зависимости ставит ячейка 6 (torch 2.6.0+cu126, torchvision 0.21.0+cu126, bitsandbytes 0.45.5 с констрейнтом; после установки сессия перезапускается автоматически — запустите ноутбук заново). LLaMA-Factory ставится ячейкой обучения с `--no-deps`.
2. Закрепите версию LLaMA-Factory: переменная окружения `LLAMAFACTORY_REF` (по умолчанию тег `v0.9.4`); фактический commit записывается в `artifacts/version.json`.
3. `FAST_DEV_RUN=True` — smoke на `manifest-smoke.jsonl` и 2 шага обучения (только проверка стека, результаты не отчётные); `FAST_DEV_RUN=False` — полный эксперимент (100 шагов, 192 тестовых примера × 2 модели).
4. Обучение и инференс возобновляемы: ячейка выбора checkpoint продолжит с последнего `checkpoint-<step>` той же конфигурации; инференс пропускает уже записанные ID.

In [16]:
save_json('environment.json', {'config': {'SEED': SEED, 'FAST_DEV_RUN': FAST_DEV_RUN, 'MODEL_ID': MODEL_ID, 'MAX_STEPS': MAX_STEPS,
                                          'IMAGE_MAX_PIXELS': IMAGE_MAX_PIXELS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS,
                                          'SCIENCEQA_REVISION': SCIENCEQA_REVISION, 'LLAMAFACTORY_REF': LF_REF, 'LLAMAFACTORY_COMMIT': commit},
                               'versions': json.loads((ARTIFACTS / 'version.json').read_text())})
ev = json.loads((ARTIFACTS / 'evaluation.json').read_text())
print('Режим:', 'ПОЛНЫЙ' if ev['full_experiment'] else 'smoke (не отчётный)')
for s, m in ev['splits'].items(): print(f"{s:13s} n={m['n']}  EM {m['exact_match_base']:.3f} -> {m['exact_match_sft']:.3f}")
print('\n'.join(ev['conclusion']))
print(sorted(p.name for p in ARTIFACTS.iterdir()))

Режим: ПОЛНЫЙ
id_test       n=64  EM 0.875 -> 0.906
ood_test      n=64  EM 0.594 -> 0.719
general_test  n=64  EM 0.859 -> 0.859
ID (natural science, 1-6 классы): +0.031; OOD (7-12 классы): +0.125; general (другие предметы): +0.000.
Прирост на OOD не меньше, чем на ID: перенос в пределах предмета сохраняется.
general не ухудшился (в пределах шума выборки).
['ATTRIBUTION.md', 'base_predictions.jsonl', 'bonus.md', 'bonus_stats.json', 'bonus_validation_predictions.jsonl', 'data_preview.png', 'data_profile.json', 'dataset_info.json', 'environment.json', 'evaluation.json', 'export_check.json', 'lab9_adapter', 'manifest.jsonl', 'manifest_check.json', 'predictions.jsonl', 'resume_protocol.json', 'sft.yaml', 'sft_predictions.jsonl', 'train.json', 'train.log', 'train_summary.json', 'validation.json', 'version.json']
